# CSCI/MATH 485 Assignment
## Customer Churn Prediction with XGBoost
## Starter Notebook

This notebook is compatible with **Jupyter Notebook** and **Google Colab**.

This starter code is only to get you started. You can change any of the existing code here to complete all the tasks.

Complete all `TODO` sections. Make sure your final submission includes:
- data analysis,
- a tuned XGBoost model,
- your chosen main evaluation metric and justification,
- interpretation of top important features,
- and a final comparison with the baseline model.

## 1. Setup

In [206]:
# xgboost is not installed.
!pip install xgboost

In [207]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

## 2. Load the Dataset

**Instructions for students**
- Load the IBM Telco Customer Churn dataset.
- Display the first few rows.
- Confirm the dataset shape.
- If the url doesn't work for you, download the csv file from the Canvas Assignment#4

In [208]:
# Load the dataset
url = "https://raw.githubusercontent.com/plotly/datasets/master/telco-customer-churn-by-IBM.csv"
telco_cus = pd.read_csv(url)
#telco_cus = pd.read_csv("telco-customer-churn-by-IBM.csv")

# Display the first 5 rows of the dataset (done below)
telco_cus.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [209]:
# Display the shape of the dataset
telco_cus.shape

(7043, 21)

## 3. Data Exploration

Complete the following:
- Print all column names
- Show data types
- Count missing values in each column
- Show the distribution of the target variable
- Write a short note: Is this a classification or regression problem? Why is this useful in business?

In [210]:
# Information on dataset
telco_cus.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [211]:
# Print column names
telco_cus.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [212]:
# Print data types
telco_cus.dtypes

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

In [213]:
# Print missing values for each column
telco_cus.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [214]:
# Target distribution
telco_cus["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [215]:
# Extra: display other information of the dataset that you think can be useful
# percentage distribution
telco_cus["Churn"].value_counts(normalize=True)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

In [216]:
# Extra: display other information of the dataset that you think can be useful
# TotalCharges column has 11 empty strings that needs to be handled.
telco_cus["TotalCharges"].value_counts()

TotalCharges
          11
20.2      11
19.75      9
20.05      8
19.9       8
          ..
6849.4     1
692.35     1
130.15     1
3211.9     1
6844.5     1
Name: count, Length: 6531, dtype: int64

**TODO (write your answer below):**

1. What kind of machine learning problem is this?

   This is a classification problem because the goal is to predict whether a customer will churn (Yes/No) which is a categorical outcome.
   
2. Why is churn prediction important in a business setting?

   Churn Prediction is important in a business context because it allows companies to proactively retain customers, reduce revenue loss, and optimize marketing strategies by focusing on high-risk customers. It also enables data-driven decision making, that helps businesses allocate resources efficiently. 



## 4. Basic Cleaning

Complete the following:
- Identify whether there is an ID column that should be removed
- Convert the target column into binary form if needed
- Convert any numeric-looking columns stored as text into numeric values

In [217]:
# Make a copy so the original data remains unchanged
clean = telco_cus.copy()

# Drop any unnecessary identifier column(s)
clean = clean.drop(columns=["customerID"])

# Convert the target column to 0/1
clean["Churn"] = clean["Churn"].map({"Yes": 1, "No": 0})

# Convert any columns that should be numeric into numeric type
clean["TotalCharges"] = pd.to_numeric(clean["TotalCharges"], errors="coerce")

# 4. Handle invalid parsing if needed
# TotalCharge now has NaN values, fill with median
#clean["TotalCharges"].fillna(clean["TotalCharges"].mean(), inplace=True)
clean.fillna({"TotalCharges": clean["TotalCharges"].median()}, inplace = True)
    
clean.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [218]:
# Identify numeric and categorical features
numeric = clean.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical = clean.select_dtypes(include=["object"]).columns.tolist()

# remove target from numeric list
numeric.remove("Churn")

print("Numeric features:", numeric)
print("\nCategorical features:", categorical)

Numeric features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [219]:
clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [220]:
# Print missing values for each column
clean.isnull().sum()

gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

## 5. Define Features and Target

Complete the following:
- Define `X` and `y`
- Set the correct target column

In [223]:
# Replace with the correct target column name
target_col = "Churn"

X = clean.drop(columns=[target_col])
y = clean[target_col]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (7043, 19)
Target shape: (7043,)


## 7. Train/Test Split

Complete the following:
- Split the dataset into training and testing sets
- Use stratification if appropriate

In [224]:
# Create train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=45,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (5634, 19)
X_test shape: (1409, 19)


## 8. Preprocessing Pipelines

Build preprocessing for:
- numeric features
- categorical features

Then combine them into a `ColumnTransformer`.

In [225]:
# Build numeric preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")), # handles missing numeric values
    ("scaler", StandardScaler())
])

# Build categorical preprocessing pipeline
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")), # handles missing categories
    ("onehot", OneHotEncoder(handle_unknown="ignore")) # convert categories to numbers 
])

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric),
    ("cat", categorical_transformer, categorical)
])

preprocessor


,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


## 9. Baseline Model: Logistic Regression

Complete the following:
- Train a Logistic Regression model as the baseline
- Generate predictions
- Evaluate using multiple metrics
- You may need to adjust `max_iter` if the model is not converging.

In [226]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Fit the baseline model
baseline_model.fit(X_train, y_train)

# Generate predicted labels
baseline_pred = baseline_model.predict(X_test)

# Generate predicted probabilities for ROC-AUC
baseline_probs = baseline_model.predict_proba(X_test)[:, 1]

In [227]:
# Compute and print: Accuracy
print("Accuracy:", accuracy_score(y_test, baseline_pred))

# Compute and print: Precision
print("Precision:", precision_score(y_test, baseline_pred))

# Compute and print: Recall
print("Recall:", recall_score(y_test, baseline_pred))

# Compute and print: F1-score
print("F1-score:", f1_score(y_test, baseline_pred))

# Compute and print: ROC-AUC
print("ROC-AUC:", roc_auc_score(y_test, baseline_probs))


Accuracy: 0.7927608232789212
Precision: 0.625
Recall: 0.5481283422459893
F1-score: 0.584045584045584
ROC-AUC: 0.8344674365134722


In [228]:
# Print classification report and confusion matrix
print("\nClassification Report:")
print(classification_report(y_test, baseline_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, baseline_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.88      0.86      1035
           1       0.62      0.55      0.58       374

    accuracy                           0.79      1409
   macro avg       0.73      0.71      0.72      1409
weighted avg       0.79      0.79      0.79      1409

Confusion Matrix:
[[912 123]
 [169 205]]


## 10. Choose Your Main Evaluation Metric

Choose **one main metric** for this churn problem.

You must explain:
- which metric you chose,
- why it is appropriate,
- and why it is more informative than accuracy alone for this problem.

**TODO (write your answer below):**

- I choose ROC-AUC as the main evaluation metrics for this churn problem.
- ROC-AUC measures how well our model can distinguish between customers who churn and those who do not churn across all classification thresholds. This is important because the dataset is imbalanced, with more non-churn customers than churn customers. It also evaluates the model's ability to rank positive cases (churn) higher than negative ones and it considers both the true positive rate (TPR) and false positive rate (FPR), making it more appropriate and informative for evaluating overall model discrimination capability in this context.
- Using accuracy alone can be misleading in imbalanced datasets because accuracy just measures the overall correctness of a model.

## 11. XGBoost Model

Complete the following:
- Build an XGBoost pipeline
- Tune at least 3 hyperparameters
- Use either `GridSearchCV` or your own tuning approach

In [229]:
xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        eval_metric="logloss",
        random_state=45
    ))
])

# Define a hyperparameter grid with at least 3 hyperparameters
param_grid = {
    "classifier__n_estimators": [50, 100, 200],
    "classifier__max_depth": [3, 5, 7],
    "classifier__learning_rate": [0.01, 0.05, 0.1],
    "classifier__subsample": [0.8, 1.0]
}

param_grid

{'classifier__n_estimators': [50, 100, 200],
 'classifier__max_depth': [3, 5, 7],
 'classifier__learning_rate': [0.01, 0.05, 0.1],
 'classifier__subsample': [0.8, 1.0]}

In [230]:
# Run GridSearchCV (or perform manual tuning)
# Choose a scoring metric that matches your selected main evaluation metric

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring="roc_auc",   # TODO: replace with your chosen scoring metric
    cv=5,
    n_jobs=-1,
    verbose=1
)
# Fit grid search on training data
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 54 candidates, totalling 270 fits


,estimator,"Pipeline(step...=None, ...))])"
,param_grid,"{'classifier__learning_rate': [0.01, 0.05, ...], 'classifier__max_depth': [3, 5, ...], 'classifier__n_estimators': [50, 100, ...], 'classifier__subsample': [0.8, 1.0]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [231]:
# Print the best hyperparameters
print("Best hyperparameters:", grid_search.best_params_)

# Save the best model
best_model = grid_search.best_estimator_

Best hyperparameters: {'classifier__learning_rate': 0.1, 'classifier__max_depth': 3, 'classifier__n_estimators': 50, 'classifier__subsample': 0.8}


## 12. Evaluate the Tuned XGBoost Model

Evaluate XGBoost using the same metrics as the baseline.

In [232]:
# Generate predictions and probabilities using the best XGBoost model
xgb_pred = best_model.predict(X_test)

# Generate predicted probabilities for ROC-AUC
xgb_probs = best_model.predict_proba(X_test)[:, 1]

In [233]:
# Compute and print: Accuracy
print("Accuracy:", accuracy_score(y_test, xgb_pred))

# Compute and print: Precision
print("Precision:", precision_score(y_test, xgb_pred))

# Compute and print: Recall
print("Recall:", recall_score(y_test, xgb_pred))

# Compute and print: F1-score
print("F1-score:", f1_score(y_test, xgb_pred))

# Compute and print: ROC-AUC
print("ROC-AUC:", roc_auc_score(y_test, xgb_probs))

Accuracy: 0.7977288857345636
Precision: 0.6518771331058021
Recall: 0.5106951871657754
F1-score: 0.5727136431784108
ROC-AUC: 0.8376902012451884


In [234]:
# Print classification report and confusion matrix
print("\nClassification Report:")
print(classification_report(y_test, xgb_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, xgb_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.65      0.51      0.57       374

    accuracy                           0.80      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409

Confusion Matrix:
[[933 102]
 [183 191]]


## 13. Feature Importance

Use the trained XGBoost model to:
- extract feature importances,
- recover transformed feature names,
- display the top 5 to 10 most important features.

In [235]:
# Access the fitted preprocessor and fitted XGBoost classifier from the pipeline
fitted_preprocessor = best_model.named_steps["preprocessor"]
fitted_xgb = best_model.named_steps["classifier"]

# Get transformed feature names
features = fitted_preprocessor.get_feature_names_out()
features

array(['num__SeniorCitizen', 'num__tenure', 'num__MonthlyCharges',
       'num__TotalCharges', 'cat__gender_Female', 'cat__gender_Male',
       'cat__Partner_No', 'cat__Partner_Yes', 'cat__Dependents_No',
       'cat__Dependents_Yes', 'cat__PhoneService_No',
       'cat__PhoneService_Yes', 'cat__MultipleLines_No',
       'cat__MultipleLines_No phone service', 'cat__MultipleLines_Yes',
       'cat__InternetService_DSL', 'cat__InternetService_Fiber optic',
       'cat__InternetService_No', 'cat__OnlineSecurity_No',
       'cat__OnlineSecurity_No internet service',
       'cat__OnlineSecurity_Yes', 'cat__OnlineBackup_No',
       'cat__OnlineBackup_No internet service', 'cat__OnlineBackup_Yes',
       'cat__DeviceProtection_No',
       'cat__DeviceProtection_No internet service',
       'cat__DeviceProtection_Yes', 'cat__TechSupport_No',
       'cat__TechSupport_No internet service', 'cat__TechSupport_Yes',
       'cat__StreamingTV_No', 'cat__StreamingTV_No internet service',
       'cat__

In [236]:
# Get feature importances
imp_features = fitted_xgb.feature_importances_
imp_features

array([0.01910802, 0.03468181, 0.01644479, 0.01049548, 0.01016398,
       0.        , 0.00053166, 0.        , 0.00957091, 0.        ,
       0.0097481 , 0.        , 0.01302578, 0.        , 0.        ,
       0.01763836, 0.10383458, 0.        , 0.06632927, 0.        ,
       0.        , 0.01117308, 0.        , 0.        , 0.00262745,
       0.        , 0.        , 0.06280482, 0.        , 0.00447553,
       0.0056937 , 0.        , 0.01386987, 0.0096841 , 0.        ,
       0.02355481, 0.4267227 , 0.01789887, 0.02173914, 0.02480087,
       0.        , 0.00734027, 0.01101892, 0.03850744, 0.00651567],
      dtype=float32)

In [237]:
# Create a DataFrame sorted by importance
sort_imp_features = pd.DataFrame({
    "feature": features,
    "importance": imp_features}
).sort_values(by="importance", ascending=False)

sort_imp_features

,feature,importance
36,cat__Contract_Month-to-month,0.426723
16,cat__InternetService_Fiber optic,0.103835
18,cat__OnlineSecurity_No,0.066329
27,cat__TechSupport_No,0.062805
43,cat__PaymentMethod_Electronic check,0.038507
1,num__tenure,0.034682
39,cat__PaperlessBilling_No,0.024801
35,cat__StreamingMovies_Yes,0.023555
38,cat__Contract_Two year,0.021739
0,num__SeniorCitizen,0.019108


In [238]:
# Display top 10 most important feature
sort_imp_features.head(10)

,feature,importance
36,cat__Contract_Month-to-month,0.426723
16,cat__InternetService_Fiber optic,0.103835
18,cat__OnlineSecurity_No,0.066329
27,cat__TechSupport_No,0.062805
43,cat__PaymentMethod_Electronic check,0.038507
1,num__tenure,0.034682
39,cat__PaperlessBilling_No,0.024801
35,cat__StreamingMovies_Yes,0.023555
38,cat__Contract_Two year,0.021739
0,num__SeniorCitizen,0.019108


## 14. Interpret the Top Features

Write a short interpretation of the most important features.

Your explanation should:
- use plain language,
- connect features to churn behavior,
- and explain what the company might learn from them.

**TODO (write your answer below):**

- Customers with month-to-month contracts show the highest churn rates. Companies should pay more attention to customers on month-to-month contracts and encourage them to switch to long-term plans, which may improve retention.
- Customers using fiber optic internet services are more likely to churn compared to those using DSL internet services and those not using internet services. This might be due to higher cost or unmet expectations regarding service quality.
- Customers who do not have online security or tech support are more likely to churn compared to those who have online security. So, improving internet security and providing more tech services may help customers feel safeguarded, increase their satisfaction and improve retention.
- Customers who pay using electronic checks are more likely to churn compared to customers who mailed checks or use automatic bank or credit card transfer. This might be associated with the fact that eCheck process is manual and sometimes creates delayed failure scenario, which may frustrate customers and increase the likelihood of churn.
- Tenure is a critical predictor of churn, customers with shorter tenure are more likely to churn compared to customers who have been with a company for long. Having tailored onboarding for newly acquired customers might improve overall retention. 


## 15. Final Comparison: Logistic Regression vs XGBoost

Compare the two models using your results.

Your discussion should answer:
- Which model performed better?
- On which metric(s)?
- Why might XGBoost perform better on this dataset?
- What is one limitation of XGBoost?

In [242]:
clean
#clean["Contract"].value_counts()
#clean["InternetService"].value_counts()
#clean["OnlineSecurity"].value_counts()
#clean["TechSupport"].value_counts()
#clean["PaymentMethod"].value_counts()
#clean["StreamingMovies"].value_counts()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,0
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,0
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,0
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,1


In [240]:
#Print a side-by-side comparison of the main metrics
#for Logistic Regression and XGBoost

comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Logistic Regression": [
        accuracy_score(y_test, baseline_pred),
        precision_score(y_test, baseline_pred),
        recall_score(y_test, baseline_pred),
        f1_score(y_test, baseline_pred),
        roc_auc_score(y_test, baseline_probs)
    ],
    "XGBoost": [
        accuracy_score(y_test, xgb_pred),
        precision_score(y_test, xgb_pred),
        recall_score(y_test, xgb_pred),
        f1_score(y_test, xgb_pred),
        roc_auc_score(y_test, xgb_probs)
    ]
})

comparison

,Metric,Logistic Regression,XGBoost
0,Accuracy,0.792761,0.797729
1,Precision,0.625000,0.651877
2,Recall,0.548128,0.510695
3,F1 Score,0.584046,0.572714
4,ROC-AUC,0.834467,0.837690


**TODO (write your answer below):**

The XGBoost model performed slightly better overall compared to Logistic Regression. It achieved higher accuracy (0.798 vs 0.793), precision (0.652 vs 0.625), and ROC-AUC (0.838 vs 0.834). However, the logistic regression model had slightly higher recall (0.548 vs 0.511), and F1-score (0.584 vs 0.573).

XGBoost may perform better for this dataset because it is a boosting algorithm that builds multiple trees sequentially, allowing it to capture complex, non-linear relationships in the data; unlike Logistic Regression, which assumes a linear relationship between features and the target. XGBoost can also model interactions between features and iteratively correct errors from previous models, leading to improved predictive performance.

One limitation of XGBoost is that it is computationally more expensive and complex compared to Logistic Regression. It also requires careful hyperparameter tuning and is less interpretable, making it harder to explain predictions in a business setting.

In practice, the choice between these models depends on the trade-off between performance and interpretability.


## 16. Suggested Submission Checklist

Before submitting, make sure your notebook includes:
- completed code cells,
- outputs for each section,
- your selected main evaluation metric and justification,
- tuned XGBoost hyperparameters,
- feature importance results,
- interpretation of important features,
- and a final comparison of the two models.
